# Personal Income Tax Walkthrough — Provincial Canada

This notebook demonstrates the **personal-income-tax (PIT) addon**: progressive, statutory
PIT layered on top of the existing provincial model as an **opt-in**, leaving the model's
default behaviour untouched when it is off.

## How to read this notebook

**Part I — Initialization** is the upstream model, unchanged. It is the same data
configuration, the same pickle, the same `SimulationConfiguration` and run loop as
`Sample_macroabm-Canada_provincial_run.ipynb`. Nothing in Part I is tax-specific; skim it.

**Part II — The tax addon** is the whole story. Everything new lives there.

## What Part II demonstrates

One calibrated economy, three scenarios. They differ **only** in which governments opt in —
`activate_progressive_pit` is a per-government flag:

| # | Scenario | Opted in | Result |
|---|----------|----------|--------|
| 01 | **Control** — no addon | nobody | all ten provinces flat |
| 02 | Progressive PIT, BC | `CAN_BC` | BC progressive; nine flat |
| 03 | Progressive PIT, all | all ten | all ten on their own schedules |

The claim, and it is exact: **with the addon off, the model is bit-for-bit identical to one
with no taxation addon at all** — even when the full taxation data is present and attached.
The addon adds a layer; it does not modify what is underneath.

---
# Part I — Initialization *(upstream model, unchanged)*

Nothing here is tax-specific. This is the standard provincial setup from the sample
notebooks: paths, the 10-province data configuration, the calibrated pickle, and the
firm/productivity parameters. Skip to Part II for the addon.

## I.0 — User settings

**These are the only two settings you should need to change.** Everything else in this
notebook is fixed by the upstream model or by the tax addon's design.

### `RAW_DATA_PATH`

The raw-data folder — it must contain `icio/`, `wiod_sea/`, `hfcs/` and `taxation/`. It ships
as the upstream placeholder; if that path does not exist but a `raw_data/` folder sits beside
this notebook, that one is used automatically.

### `SCALE`

How many real households each synthetic household represents. **Lower means more agents** — a
finer-grained economy, at the cost of time and disk.

| `SCALE` | BC households | pickle build | 3 sim runs | total | results file |
|---|---|---|---|---|---|
| **1000** | 1,874 | 2 min | 5 min | ~7 min | 6.7 GB |
| 500 | 3,748 | 4 min | 7 min | ~11 min | ~13 GB † |
| 250 | 7,497 | 6 min | 12 min | ~18 min | ~27 GB † |

Times rounded to the nearest minute on a quiet machine; the notebook runs three simulations.
All build and run times are **CPU-bound and vary with machine load** — a busy machine can
easily double them. († results-file sizes for 500/250 are estimates — the household series
dominate the file, so it roughly doubles each time `SCALE` halves; only the 1000 size is
measured.)

Three things worth knowing before you change it:

- **The build cost is dominated by population synthesis**, which grows with the agent count —
  but sub-linearly, so halving `SCALE` (doubling the agents) less than doubles the build time.
  The data-read cost is a small fixed base and is cache-insensitive: a cold build is no slower
  than a warm one (measured).
- **`SCALE` is baked into the pickle at build time**, so changing it forces a rebuild. The
  pickle filename carries the scale, so an old one is never silently reused.
- **`SCALE` shifts BC's effective tax rate**, and not because of any flaw in the tax code —
  that is verified scale-homogeneous (normalised per-household income distributions coincide
  across scales: same median, same Gini; only the top percentile moves). A progressive tax is
  dominated by the **top tail** of the income distribution, and the tail is sampled differently
  at different agent counts.

  Indicative BC effective rate — *one machine, one commit; do not treat as fixed*:
  `SCALE=1000` ≈ 8.25–8.33%.

  The figures once quoted here for `SCALE=500` (≈4.0%) and `SCALE=250` (≈4.1%) were measured
  **before income was annualized for assessment**, when a quarter's income met the annual
  brackets directly. Every rate from that period reads roughly half of what the schedule
  implies, so they are not comparable to the figure above and have not been re-measured.

  The spread across repeated builds at a **fixed** scale is of the same order as the spread
  **across** scales, so the two cannot be separated from single measurements. Build-to-build
  reproducibility is under active investigation in the base model. Until it settles: calibrate
  against **several builds**, never one, and do not carry a tuned level across `SCALE`.

In [ ]:
# ==================================================================================
#  USER SETTINGS  --  see I.0 above for what these do and what they cost
# ==================================================================================

# Raw-data folder. Falls back to ./raw_data if this placeholder does not exist.
RAW_DATA_PATH = r"path/to/raw_data"

# Real households per synthetic household. Lower = more agents = slower, bigger.
# Changing this forces a pickle rebuild. 1000 is the upstream sample's value.
SCALE = 1000

# ==================================================================================

## I.1 — Autoreload, imports, paths

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import logging
import time
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# --- quieten known, benign noise so the output stays readable ---------------------
# EVERYTHING silenced here is UPSTREAM and reproduces in the unmodified sample
# notebooks with no taxation data at all. None of it comes from the tax addon, and
# none of it reaches the results: GDP, CPI and production are finite for every
# province (II.8 reads them back from the results file). Delete this block to see it all.

# (1) Non-finite values. Eight of the ten provinces have an industry with zero output
#     in the 2014 provincial IO table (Manitoba: fishing; PEI: iron and steel; ...),
#     so its technical-coefficient column is computed as input/0 = inf; and firms with
#     zero wages divide by zero during synthetic matching. Note the pattern is
#     deliberately broad -- numpy raises "in divide", "in scalar divide" and
#     "in multiply", and a filter matches from the START of the message.
warnings.filterwarnings("ignore", category=RuntimeWarning,
                        message="invalid value encountered")
warnings.filterwarnings("ignore", category=RuntimeWarning,
                        message="divide by zero encountered")

# (2) pandas housekeeping from the HFCS and ICIO readers: mixed dtypes on read, and a
#     fragmented DataFrame built by repeated inserts.
try:
    from pandas.errors import DtypeWarning, PerformanceWarning
    warnings.filterwarnings("ignore", category=DtypeWarning)
    warnings.filterwarnings("ignore", category=PerformanceWarning)
except ImportError:
    pass

# (3) "Overwriting Consumption Weights by Income with French Data" -- emitted once per
#     province because the provinces are France-proxied for the household finance
#     survey, which is the intended upstream configuration (see I.2).
class _DropUpstreamNoise(logging.Filter):
    NOISE = ("Overwriting Consumption Weights",)

    def filter(self, record):
        msg = record.getMessage()
        return not any(n in msg for n in self.NOISE)


logging.getLogger().addFilter(_DropUpstreamNoise())

# (4) The tax-parameter scalar fallback. EXPECTED, not a defect -- see II.3. The four
#     scalars in tax_parameters.yaml are deliberately year-invariant, so any year after
#     the 2014 block reuses it. The reader logs this once per fallback block.
logging.getLogger(
    "macromodel.configurations.tax_parameters.tax_parameters_reader"
).setLevel(logging.ERROR)

import macro_data
from macro_data import DataWrapper, configuration_utils
from macro_data.configuration.countries import Country as CountryCode
from macro_data.configuration.region import Region
from macromodel.configurations import SimulationConfiguration, CountryConfiguration
from macromodel.simulation import Simulation

# Resolve the two user settings from I.0.
INPUT_PATH       = Path(RAW_DATA_PATH)
BASE_DIR         = Path.cwd()
OUTPUT_DIRECTORY = BASE_DIR / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# If RAW_DATA_PATH is still the placeholder (or does not exist) but a raw_data/ folder
# sits beside this notebook, use that.
if not INPUT_PATH.exists() and (BASE_DIR / "raw_data").exists():
    INPUT_PATH = BASE_DIR / "raw_data"
    print(f"RAW_DATA_PATH not found; using {INPUT_PATH}")

PROVINCE_INFO = {
    "CAN_AB": "Alberta",                   "CAN_BC": "British Columbia",
    "CAN_MB": "Manitoba",                  "CAN_NB": "New Brunswick",
    "CAN_NL": "Newfoundland and Labrador", "CAN_NS": "Nova Scotia",
    "CAN_ON": "Ontario",                   "CAN_PE": "Prince Edward Island",
    "CAN_QC": "Quebec",                    "CAN_SK": "Saskatchewan",
}
PROVINCES   = list(PROVINCE_INFO)
N_PROVINCES = len(PROVINCES)

print(f"Input path : {INPUT_PATH}  exists={INPUT_PATH.exists()}")
print(f"Provinces  : {N_PROVINCES}")

## I.2 — Data configuration and the calibrated pickle

The standard 10-province configuration: France-proxied household finance survey, one
representative firm per sector, quarterly timesteps. `SCALE` comes from **I.0**; everything
else here is fixed.

**Note what is absent: nothing here mentions taxation.** The schedules are discovered from
the raw-data tree, like the optional energy-sector readers. The one tax-related argument,
`taxation_filenames`, is introduced in Part II — it selects *which* schedule files to read.

In [ ]:
# --- upstream data configuration ------------------------------------------------
def build_data_config():
    cfg = configuration_utils.default_data_configuration(
        countries=["CAN"],
        aggregate_industries=False,
        proxy_country_dict={"CAN": "FRA"},
        use_disagg_can_2014_reader=True,
    )
    cfg.year = 2014
    cfg.time_unit = 3          # quarterly
    cfg.can_disaggregation = True
    cfg.aggregate_industries = False
    cfg.prune_date = None
    cfg.seed = 0

    base = cfg.country_configs[CountryCode("CAN")]
    base.single_firm_per_industry = True
    base.single_bank = True
    base.single_government_entity = True
    base.firms_configuration.constructor = "Default"
    base.scale = SCALE

    provinces = [Region.from_code(c, n) for c, n in PROVINCE_INFO.items()]
    for province in provinces:
        cfg.country_configs[province] = base
        cfg.country_configs[province].eu_proxy_country = CountryCode("FRA")
    cfg.aggregation_structure = {CountryCode("CAN"): provinces}
    return cfg


# --- upstream simulation parameters ---------------------------------------------
TIMESTEPS  = 16            # 16 quarters = 2014-2018
SEED       = 0
START_YEAR = 2014

tfp_base_growth_rate                            = 0.001
tfp_investment_elasticity                       = 0.5
productivity_growth_investment_effectiveness    = 0.3
technical_coefficients_investment_effectiveness = 0.3
diminishing_returns_factor                      = 0.1
hurdle_rate                                     = 0.01
investment_effectiveness                        = 0.3
technical_investment_effectiveness              = 0.3
technical_diminishing_returns                   = 0.1
tfp_investment_share                            = 0.5
max_investment_fraction                         = 0.2


# --- upstream per-province firm wiring ------------------------------------------
def apply_upstream_config(cfg, province):
    """Everything the upstream sample notebook sets. No tax here."""
    cc = cfg.country_configurations[province]

    cc.firms.functions.productivity_investment_planner.name = "SimpleProductivityInvestmentPlanner"
    cc.firms.functions.productivity_investment_planner.parameters.update({
        "n_firms":                            n_industries,
        "tfp_investment_share":               tfp_investment_share,
        "max_investment_fraction":            max_investment_fraction,
        "investment_effectiveness":           investment_effectiveness,
        "technical_investment_effectiveness": technical_investment_effectiveness,
        "technical_diminishing_returns":      technical_diminishing_returns,
        "hurdle_rate":                        hurdle_rate,
    })
    cc.firms.functions.productivity_growth.name = "SimpleTFPGrowth"
    cc.firms.functions.productivity_growth.parameters = {
        "investment_effectiveness": productivity_growth_investment_effectiveness,
    }
    cc.firms.parameters.tfp_base_growth_rate      = tfp_base_growth_rate
    cc.firms.parameters.tfp_investment_elasticity = tfp_investment_elasticity

    cc.firms.functions.technical_coefficients_growth.name = "SimpleTechnicalGrowth"
    cc.firms.functions.technical_coefficients_growth.parameters = {
        "investment_effectiveness":   technical_coefficients_investment_effectiveness,
        "diminishing_returns_factor": diminishing_returns_factor,
    }


print(f"Timesteps : {TIMESTEPS}  ({TIMESTEPS // 4} years from {START_YEAR})")
print("Upstream configuration ready. Everything above is the standard provincial model.")

---
# Part II — The tax addon

Everything below is new. The addon engages only when **two gates** are open:

- **Gate 1 — data.** The province's `SyntheticCountry` carries a `TaxationReader`.
  `TaxationStore` reads the jurisdictions straight from the bracket file's `jurisdiction` column and
  hands each country its own (`CAN_BC` -> `bc`). A jurisdiction the data does not cover gets
  `None` and runs the flat Income Tax rate — a normal outcome, not an error.
- **Gate 2 — opt-in.** The government sets `central_government.activate_progressive_pit`,
  which defaults to **`False`**.

Either gate closed => the unchanged flat configuration.

## II.1 — Gate 1: the data decides who *can* be taxed

The reader is bound to the consolidated, jurisdiction-keyed *schema*, not to particular filenames. So
**which schedule files a run reads is how it chooses its tax coverage** — no code change.

This cell reads the schedules directly (no simulation) to show the point: the canonical files
cover BC and the federal government; the comprehensive files cover all ten provinces.

In [ ]:
from macro_data.readers.taxation import TaxationStore

TAXATION_PATH = INPUT_PATH / "taxation" / "personal_income_tax"

CANONICAL     = {}                                    # {} = default filenames: BC + federal
COMPREHENSIVE = {
    "rates":    "rates_thresholds_new_provinces.csv",
    "credits":  "non_refundable_tax_credits_new_jurisdictions.csv",
    "dividend": "dividend_tax_credit_schedule_new_jurisdictions.csv",
}

if not TAXATION_PATH.exists():
    raise FileNotFoundError(
        f"No taxation schedules at {TAXATION_PATH}.\n"
        f"Set INPUT_PATH in cell I.1 to your raw-data folder — it must contain "
        f"taxation/personal_income_tax/. Without it the addon cannot be "
        f"demonstrated (the model would simply run flat)."
    )

HAVE_COMPREHENSIVE = (TAXATION_PATH / COMPREHENSIVE["rates"]).exists()

store_canonical = TaxationStore.from_dir(TAXATION_PATH, **CANONICAL)
store_all = (TaxationStore.from_dir(TAXATION_PATH, **COMPREHENSIVE)
             if HAVE_COMPREHENSIVE else None)

print(f"{'province':<10} {'canonical files':>17} {'comprehensive files':>21}")
print("-" * 50)
for p in PROVINCES:
    a = store_canonical.for_country(p)
    b = store_all.for_country(p) if store_all else None
    print(f"{p:<10} {(a.jurisdiction if a else 'flat'):>17} "
          f"{((b.jurisdiction if b else 'flat') if store_all else 'n/a'):>21}")

print(f"\ncanonical     covers : {sorted(store_canonical.jurisdictions)}")
if store_all:
    print(f"comprehensive covers : {sorted(store_all.jurisdictions)}")
print("\nCoverage follows the DATA, not the code. No province is ever taxed on "
      "another's schedule.")

## II.2 — Build the pickle

One pickle, built with the comprehensive schedules so that **every** province carries its own
statutory brackets and credits. Each government then opts in — or does not — independently,
which is what lets all three scenarios run against a single calibrated economy.

`taxation_filenames` is the only tax-related argument in the whole build.

> **Stale-pickle warning.** `SyntheticCountry` carries a `taxation` field. A pickle built
> before the addon existed unpickles happily *without* it and the model then runs flat — no
> error, just no progressive PIT. The guard regenerates rather than trusting that the file
> merely exists.

In [ ]:
PKL_PATH    = OUTPUT_DIRECTORY / f"data_provincial_pit_all_scale{SCALE}.pkl"
H5_FILENAME = "results_pit_provincial.h5"     # the standard upstream results artifact
SCHEDULES = COMPREHENSIVE if HAVE_COMPREHENSIVE else CANONICAL


def pickle_is_usable(path: Path) -> bool:
    # A pickle is only usable if it carries the taxation field AND still satisfies the
    # model's GDP identity checks. Two ways a stale pickle bites:
    #   1. Built before the tax addon -> no `taxation` field; the model silently runs FLAT.
    #   2. Built before a model fix that changed the DATA layer (e.g. the social-housing
    #      rent exclusion) -> it unpickles fine, then dies inside from_datawrapper with
    #      "AssertionError: mismatch, output/income GDP", which looks like a model bug
    #      rather than a stale file. Constructing a throwaway Simulation catches it HERE,
    #      with an explanation, instead of there.
    try:
        probe = DataWrapper.init_from_pickle(path)
    except Exception:
        return False
    if not all(hasattr(c, "taxation") for c in probe.synthetic_countries.values()):
        print("   pickle predates the taxation field")
        return False
    try:
        cfg = SimulationConfiguration(
            seed=SEED,
            country_configurations={
                p: CountryConfiguration.n_industry_default(n_industries=probe.n_industries)
                for p in PROVINCES
            },
            t_max=1,
        )
        Simulation.from_datawrapper(datawrapper=probe, simulation_configuration=cfg)
    except AssertionError as error:
        print(f"   pickle predates a model fix (GDP identity fails: {error})")
        return False
    return True


if PKL_PATH.exists() and pickle_is_usable(PKL_PATH):
    print(f"Reusing {PKL_PATH.name}")
    data = DataWrapper.init_from_pickle(PKL_PATH)
else:
    t0 = time.time()
    creator = DataWrapper.from_config(
        configuration=build_data_config(),
        raw_data_path=INPUT_PATH,
        single_hfcs_survey=True,
        taxation_filenames=SCHEDULES,        # <- the addon's only build-time argument
    )
    creator.save(PKL_PATH)
    print(f"Built {PKL_PATH.name} in {(time.time() - t0) / 60:.1f} min")
    data = DataWrapper.init_from_pickle(PKL_PATH)

n_industries = data.n_industries

attached = sorted(str(k) for k, v in data.synthetic_countries.items()
                  if getattr(v, "taxation", None) is not None)
print(f"n_industries          : {n_industries}")
print(f"provinces with schedules : {len(attached)}/{N_PROVINCES}")

## II.3 — Gate 2: the per-government opt-in

`activate_taxation` is the seam that consumes the flag:

```python
if not base_config.activate_progressive_pit or taxation_reader is None:
    return base_config          # <- the SAME object. This is the parity guarantee.
return build_central_government_configuration(...)
```

Parity is therefore **structural, not empirical**: with the flag off the function returns the
configuration object it was given. There is no separate flat code path that could drift.

The scenario runner below is the upstream wiring plus **one line**.

> **On the tax-parameter scalars.** `tax_parameters.yaml` carries four scalars per
> jurisdiction (dividend integration, the two dividend small-business shares, and the couple
> rental-income split). They are **modelling assumptions, not statutory figures, and are
> deliberately year-invariant** — so the file holds a single 2014 block and every later year
> reuses it. The reader logs a "not found ... falling back to 2014" message when it does so.
> That is the intended behaviour, not a missing-data defect; the message is silenced in I.1
> so it does not clutter the run. The year-varying quantities — brackets, credits, dividend
> rates — live in the schedule CSVs and *do* advance with the calendar year.

In [ ]:
import h5py

# ---------------------------------------------------------------------------------
#  What goes into the HDF5 results file.
#
#  `sim.save()` writes EVERYTHING: rest-of-world, the goods market, and all ten
#  provinces including household-level series. That is ~6.7 GB for a 16-step run --
#  ROW alone is 2.4 GB and the provincial `households` groups another ~5 GB. The
#  upstream sample writes exactly this; it is faithful, but heavy.
#
#  DEFAULT: the FULL artifact -- every province, exactly as the upstream sample
#  writes it. The tax addon is BC-calibrated, but the model is not a BC model, and a
#  BC-only results file would quietly discard the nine provinces that scenario 03
#  exists to demonstrate. Faithfulness to the upstream artifact wins over disk.
#
#  Set SAVE_FULL_H5 = False for a compact alternative (BC in full + an aggregated
#  CAN group, ~20x smaller) if disk or transfer size is a constraint.
# ---------------------------------------------------------------------------------
SAVE_FULL_H5 = True

# Series summed into the aggregated CAN group. EXTENSIVE quantities only -- a sum
# across provinces is meaningful for output, spending and revenue. Intensive ones
# (CPI, unemployment rate, price indices) are deliberately EXCLUDED: summing an index
# across provinces is meaningless, and a reader could easily mistake such a column for
# a national figure. Read those per province instead.
CAN_AGGREGATES = {
    "economy": [
        "gdp_output", "gdp_income", "gdp_expenditure",
        "total_output", "total_gross_value_added",
        "total_household_fce", "total_compensation_of_employees",
        "total_exports", "total_imports", "total_gross_fixed_capital_formation",
    ],
    "central_government": ["taxes_income"],
    "firms": ["production"],
}


def save_results(sim, file_name: str) -> None:
    path = OUTPUT_DIRECTORY / file_name

    if SAVE_FULL_H5:
        sim.save(save_dir=OUTPUT_DIRECTORY, file_name=file_name)
        size = path.stat().st_size / 1e9
        print(f"    FULL results written: {file_name}  ({size:.2f} GB)")
        return

    with h5py.File(path, "w") as f:
        sim.save_random_seed(f)
        sim.save_configuration(f)

        # British Columbia, in full -- the same group the upstream file carries.
        sim.countries["CAN_BC"].save_to_h5(f)

        # Aggregated Canada: extensive series summed across the ten provinces.
        can = f.create_group("CAN")
        for agent, keys in CAN_AGGREGATES.items():
            group = can.create_group(agent)
            for key in keys:
                try:
                    stacked = [
                        np.asarray(getattr(sim.countries[p], agent).ts.dicts[key],
                                   dtype=float)
                        for p in PROVINCES
                    ]
                except KeyError:
                    continue
                group.create_dataset(key, data=np.sum(stacked, axis=0))

    size = path.stat().st_size / 1e6
    print(f"    results written: {file_name}  ({size:,.0f} MB, BC + aggregated CAN)")

In [ ]:
# Macro aggregates recorded alongside the tax observables. The effective PIT rate is
# not just a revenue figure -- it is the scalar the model uses for wage-setting and
# for household consumption decisions -- so a change in it should propagate into
# these. They come from the same `country.<agent>.ts` series the HDF5 results file
# stores; collecting them in memory avoids writing four multi-hundred-MB files.
MACRO_SERIES = {
    "consumption":  ("economy", "total_household_fce",             "Household consumption"),
    "wages":        ("economy", "total_compensation_of_employees", "Employee compensation"),
    "unemployment": ("economy", "unemployment_rate",               "Unemployment rate"),
    "cpi":          ("economy", "cpi",                             "CPI"),
}


def run_scenario(label: str, opted_in: list[str], save_h5: str | None = None) -> dict:
    """Run one scenario. `opted_in` lists the provinces whose government opts in."""
    print(f"{label}   opted in: {opted_in if opted_in else 'nobody'}")
    cfg = SimulationConfiguration(
        seed=SEED,
        country_configurations={
            p: CountryConfiguration.n_industry_default(n_industries=n_industries)
            for p in PROVINCES
        },
        t_max=TIMESTEPS,
    )
    for province in PROVINCES:
        apply_upstream_config(cfg, province)                       # upstream, unchanged
        cfg.country_configurations[province].central_government \
            .activate_progressive_pit = province in opted_in       # <- the addon: one line

    sim = Simulation.from_datawrapper(datawrapper=data, simulation_configuration=cfg)

    ts_gdp   = np.empty((TIMESTEPS, N_PROVINCES))
    ts_pit   = np.empty((TIMESTEPS, N_PROVINCES))
    ts_settle = np.empty((TIMESTEPS, N_PROVINCES))
    ts_rate  = np.empty((TIMESTEPS, N_PROVINCES))
    ts_macro = {k: np.empty((TIMESTEPS, N_PROVINCES)) for k in MACRO_SERIES}
    ts_price = np.empty((TIMESTEPS, n_industries))   # good prices, BC as reference

    t0 = time.time()
    for t in range(TIMESTEPS):
        print(f"    timestep {t + 1}/{TIMESTEPS}", end="\r")
        sim.iterate(t)
        for pi, province in enumerate(PROVINCES):
            country = sim.countries[province]
            ts_gdp[t, pi] = country.economy.ts.current("gdp_output")[-1]
            ts_pit[t, pi] = country.central_government.ts.current("taxes_income")[0]
            # The year-end settlement is netted into taxes_income above, where it
            # is far too small to see; on its own series it is legible.
            ts_settle[t, pi] = country.central_government.ts.current(
                "pit_year_end_settlement"
            )[0]
            # The blended effective rate the government charges. NOTE: this same
            # scalar feeds wage-setting and consumption, so it is a behavioural
            # parameter, not merely a revenue statistic -- which is why the macro
            # aggregates below are worth watching.
            ts_rate[t, pi] = float(country.central_government.states["Income Tax"])

            for key, (agent, series, _lbl) in MACRO_SERIES.items():
                ts_macro[key][t, pi] = float(
                    np.asarray(getattr(country, agent).ts.current(series)).ravel()[-1]
                )
        ts_price[t] = np.asarray(
            sim.countries["CAN_BC"].economy.ts.current("good_prices"), dtype=float
        )

    effective_rate = {p: float(ts_rate[-1, pi]) for pi, p in enumerate(PROVINCES)}

    # The standard upstream results artifact, written only for the scenario asked for.
    # SCOPED by default -- see save_results() -- because the full file is ~6.7 GB.
    if save_h5:
        save_results(sim, save_h5)

    print(f"    done in {(time.time() - t0) / 60:.1f} min")
    return {"label": label, "ts_gdp": ts_gdp, "ts_pit": ts_pit, "ts_rate": ts_rate,
            "ts_settle": ts_settle,
            "ts_price": ts_price, "macro": ts_macro, "effective_rate": effective_rate}

## II.4 — The three scenarios

One economy, three flag patterns. Nothing else differs.

In [ ]:
res_01 = run_scenario("Scenario 01 - control (no addon)",        opted_in=[])
res_02 = run_scenario("Scenario 02 - progressive PIT (BC)",      opted_in=["CAN_BC"],
                      save_h5=H5_FILENAME)
res_03 = run_scenario("Scenario 03 - progressive PIT (all)",     opted_in=PROVINCES)

## II.5 — Parity: the model is unchanged when the addon is off *(verified)*

With every government opted out, the model is **bit-for-bit identical** to one carrying no
taxation data at all — a `0.000000e+00` GDP difference for every province, at every timestep.
This was verified directly during development, on the same build used here.

The result is structural, not a coincidence. `activate_taxation` short-circuits when the flag
is off and **returns the government's configuration object unchanged** — no extra code runs and
no extra random draws are consumed. There is no separate flat code path that could drift.
Merging the addon and leaving it off therefore changes nothing.

## II.6 — What the addon changes, and what it does not

**The tax treatment.** A province that has not opted in keeps the flat Income Tax rate,
exactly. Compare the effective rates across scenarios: every non-opted-in province sits on
0.09 to four decimal places, in every scenario.

**A caution about comparing GDP paths across scenarios.** The provinces are *not* isolated
from one another in a comparison like this, for a reason that has nothing to do with tax:
`Simulation` seeds the **global** NumPy random stream (`simulation.py`), so all ten provinces
draw from one shared generator. When BC's tax changes, BC consumes a different number of
random draws, and every province drawing after it receives different random numbers. The
non-BC provinces therefore *do* move between scenarios 01 and 02 — but they move because the
random stream desynchronised, not because their tax changed.

Read the **effective rate** table below, not the GDP differences, to see what the tax did.
The parity claim in II.5 is unaffected by this: with the flag off no extra draws are consumed,
the stream stays in lockstep, and the addon-off model is identical.

In [ ]:
print("Effective Income Tax rate at end of run")
print(f"   {'province':<10} {'01 control':>12} {'02 BC':>10} {'03 all':>10}")
print("   " + "-" * 44)
for province in PROVINCES:
    r1 = res_01["effective_rate"][province]
    r2 = res_02["effective_rate"][province]
    r3 = res_03["effective_rate"][province]
    tag = "  <-- opted in (02 and 03)" if province == "CAN_BC" else ""
    print(f"   {province:<10} {r1:>12.4%} {r2:>10.4%} {r3:>10.4%}{tag}")

print()
print("Scenario 02: only BC departs from the flat 0.09 - the other nine are untouched.")
print("Scenario 03: every province is on its own statutory schedule.")

## II.7 — Results

British Columbia is the calibrated jurisdiction, so it leads. The first figure is BC alone;
the province-wide charts that follow keep BC highlighted and mute the rest, which are
included to show the mechanism generalises — not because their levels are calibrated.

In [ ]:
PROV_LABELS = dict(PROVINCE_INFO)
x     = np.arange(TIMESTEPS)
runs  = [res_01, res_02, res_03]
BC    = PROVINCES.index("CAN_BC")

BC_COLOUR    = "#c0392b"        # BC: strong
MUTED        = "#b8c4d0"        # the other nine: recessive
SCEN_COLOURS = ["#7f8c8d", "#c0392b", "#2e86c1"]   # control / BC / all


def qoq_growth(series):
    """Quarter-on-quarter percentage change of a 1-D series."""
    series = np.asarray(series, dtype=float).ravel()
    return 100.0 * (series[1:] / series[:-1] - 1.0)

In [ ]:
# ================= BC — the calibrated jurisdiction ==============================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# BC effective rate over the run, by scenario
ax = axes[0]
for k, res in enumerate(runs):
    ax.plot(x, res["ts_rate"][:, BC], linewidth=2.2, color=SCEN_COLOURS[k],
            label=res["label"])
ax.axhline(0.09, color="black", linewidth=1.0, linestyle="--",
           label="flat Income Tax rate (0.09)")
ax.set_title("British Columbia — effective Income Tax rate\n"
             "the rate that also drives wage-setting and consumption", fontsize=10)
ax.set_xlabel("Timestep (3 months)")
ax.set_ylabel("Effective Income Tax rate")
ax.set_ylim(0, 0.10)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# BC PIT revenue over the run, by scenario
ax = axes[1]
for k, res in enumerate(runs):
    ax.plot(x, res["ts_pit"][:, BC], linewidth=2.2, color=SCEN_COLOURS[k],
            label=res["label"])
ax.set_title("British Columbia — personal income tax revenue", fontsize=10)
ax.set_xlabel("Timestep (3 months)")
ax.set_ylabel("PIT revenue (model units)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.suptitle("British Columbia: what the addon changes", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "pit_bc_headline.png", dpi=120)
plt.show()

print(f"BC effective rate, end of run:")
for res in runs:
    print(f"   {res['label']:<40} {res['effective_rate']['CAN_BC']:.4%}")

## The year-end filing

Withholding assesses each quarter as though the year ran at that rate, so a taxpayer whose
income moved through the year has paid the wrong amount by the time the year closes. At the
first period of the new year the tax owed on the year's *actual* income is compared with what
was withheld, and the difference settles.

The settlement is already inside `taxes_income`, but it is a fraction of a percent of the
year's liability — invisible against the revenue line. Recorded on its own series it is
legible. **Negative is a refund paid out at the filing; positive is an amount collected there.**
Both settle at the same moment — a filing has one outcome. The control scenario is flat-taxed and never files, so it is a clean
zero throughout.

A convex schedule can only ever over-withhold on the tax leg (Jensen's inequality), so refunds
are the expected direction and a collection is the exception worth looking at.


In [ ]:
# ================= BC — the year-end filing ====================================
from macromodel.sim_calendar import steps_per_year

F = int(steps_per_year())
FILINGS = list(range(F + 1, TIMESTEPS + 1, F))   # first period of each new year

fig, ax = plt.subplots(figsize=(12, 4))
width = 0.38
steps = np.arange(1, TIMESTEPS + 1)
for k, res in enumerate((res_02, res_03)):
    ax.bar(steps + (k - 0.5) * width, res["ts_settle"][:, BC], width,
           color=SCEN_COLOURS[k + 1], label=res["label"])
for q in FILINGS:
    ax.axvline(q, color="grey", linestyle=":", linewidth=1, zorder=0)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("British Columbia — year-end settlement "
             "(negative = refund paid out, positive = collection received)", fontsize=11)
ax.set_xlabel("Timestep (3 months); dotted lines are filing periods")
ax.set_ylabel("model units")
ax.legend(fontsize=8)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "pit_year_end_settlement.png", dpi=120)
plt.show()

# Each filing against the year it settles, as a share of that year's PIT revenue.
print(f"{'filing at':>10} {'settles':>9} {'settlement':>16} {'year PIT':>16} {'share':>9}")
print("-" * 64)
for q in FILINGS:
    year_slice = slice(q - 1 - F, q - 1)          # the four periods just closed
    settled = res_03["ts_settle"][q - 1, BC]
    year_pit = res_03["ts_pit"][year_slice, BC].sum()
    share = (settled / year_pit * 100) if year_pit else float("nan")
    direction = "refund" if settled < 0 else ("collection" if settled > 0 else "exact")
    print(f"{q:>10} {2014 + (q - 1) // F - 1:>9} {settled:>16,.1f} "
          f"{year_pit:>16,.1f} {share:>8.3f}%   {direction}")

control = np.abs(res_01["ts_settle"][:, BC]).max()
print(f"\ncontrol scenario, largest settlement of any period: {control:,.4f}"
      "   (flat tax never files)")


### BC — the macro aggregates the tax rate feeds into

The effective PIT rate is not merely what the government collects: the model uses that same
scalar in **wage-setting** and in **household consumption** decisions. These are the
aggregates it touches, shown across the three scenarios.

They are the same series the HDF5 results file stores (`<province>/economy/<variable>`),
collected in memory here rather than written to disk.

*These panels show the quantities the tax rate influences. Read them as context rather than a
precise measurement of the effect's size — II.9 explains why a single run does not pin down a
macro number in this model.*

In [ ]:
# ================= BC — macro transmission of the tax change =====================
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, (key, (_agent, _series, title)) in zip(axes.ravel(), MACRO_SERIES.items()):
    for k, res in enumerate(runs):
        ax.plot(x, res["macro"][key][:, BC], linewidth=2.0, color=SCEN_COLOURS[k],
                label=res["label"])
    ax.set_title(f"British Columbia — {title}", fontsize=10)
    ax.set_xlabel("Timestep (3 months)")
    ax.grid(True, alpha=0.3)
    if key == "unemployment":
        ax.set_ylabel("rate")
    elif key == "cpi":
        ax.set_ylabel("index")
    else:
        ax.set_ylabel("model units")

axes[0, 0].legend(fontsize=7)
plt.suptitle("British Columbia: how the progressive PIT reaches the wider economy",
             fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "pit_bc_macro.png", dpi=120)
plt.show()

# End-of-run summary, with a measured noise floor.
#
# BC's TAX TREATMENT IS IDENTICAL in scenarios 02 and 03 -- BC is opted in either
# way; only the other nine provinces differ. So any BC difference between 02 and 03
# is NOT BC's own tax: it is the shared-RNG wander (II.6) plus genuine interprovincial
# spillover. That gives an honest floor. A "tax effect" (01 -> 02) smaller in magnitude
# than that floor should not be read as an effect at all. The verdict below is stricter
# still: it demands the effect exceed THREE times the floor before it is called an effect.
print(f"{'series':<24} {'01 control':>13} {'02 BC':>13} "
      f"{'tax effect':>11} {'noise floor':>12}")
print("-" * 78)
for key, (_a, _s, title) in MACRO_SERIES.items():
    a = res_01["macro"][key][-1, BC]
    b = res_02["macro"][key][-1, BC]
    c = res_03["macro"][key][-1, BC]
    effect = (b / a - 1) * 100 if a else float("nan")
    floor  = abs((c / b - 1) * 100) if b else float("nan")
    verdict = "" if abs(effect) > 3 * floor else "   <-- within noise"
    print(f"{title:<24} {a:>13,.4g} {b:>13,.4g} "
          f"{effect:>10.2f}% {floor:>11.2f}%{verdict}")

print("\n'tax effect'  = 01 -> 02 : BC opts in (this is the tax).")
print("'noise floor' = 02 -> 03 : BC's tax is UNCHANGED between these two runs, so this")
print("                           is shared-RNG wander plus interprovincial spillover.")

In [ ]:
# ============ All provinces — control vs all-progressive =========================
# Scenario 02 (BC only) is omitted here: BC's move is already in the headline above, and
# on a by-province chart its bars would just duplicate the control for the other nine.
rate_runs = [res_01, res_03]
fig, ax = plt.subplots(figsize=(12, 5))
pos   = np.arange(N_PROVINCES)
width = 0.8 / len(rate_runs)

for k, res in enumerate(rate_runs):
    rates   = [res["effective_rate"][p] for p in PROVINCES]
    colours = [BC_COLOUR if p == "CAN_BC" else MUTED for p in PROVINCES]
    ax.bar(pos + (k - (len(rate_runs) - 1) / 2) * width, rates, width=width,
           color=colours, edgecolor="white", linewidth=0.4,
           hatch=[None, "//"][k], label=res["label"])

ax.axhline(0.09, color="black", linewidth=1.0, linestyle="--",
           label="flat Income Tax rate (0.09)")
ax.set_xticks(pos)
labels = ax.set_xticklabels([p.replace("CAN_", "") for p in PROVINCES])
labels[BC].set_fontweight("bold")
labels[BC].set_color(BC_COLOUR)
ax.set_ylabel("Effective Income Tax rate")
ax.set_title("Effective Income Tax rate by province: no addon vs all-provinces progressive\n"
             "BC is the calibrated jurisdiction; the other nine demonstrate the "
             "mechanism, not calibrated levels", fontsize=10)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "pit_effective_rate.png", dpi=120)
plt.show()

In [ ]:
# --- GDP level (top row) and quarter-on-quarter growth (bottom), BC highlighted --
fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True, sharey="row")
for col, res in enumerate(runs):
    for pi, province in enumerate(PROVINCES):
        is_bc = province == "CAN_BC"
        style = dict(linewidth=2.4 if is_bc else 0.9,
                     color=BC_COLOUR if is_bc else MUTED,
                     alpha=1.0 if is_bc else 0.7, zorder=3 if is_bc else 1)
        axes[0, col].plot(x, res["ts_gdp"][:, pi],
                          label=PROV_LABELS[province] if is_bc else None, **style)
        axes[1, col].plot(x[1:], qoq_growth(res["ts_gdp"][:, pi]), **style)
    axes[0, col].set_title(res["label"], fontsize=9)
    axes[1, col].axhline(0.0, color="black", linewidth=0.7, linestyle="--")
    axes[1, col].set_xlabel("Timestep (3 months)")
    for row in (0, 1):
        axes[row, col].grid(True, alpha=0.3)
axes[0, 0].set_ylabel("Nominal GDP output (model units)")
axes[1, 0].set_ylabel("GDP growth, quarter-on-quarter (%)")
axes[0, 0].legend(fontsize=8)
plt.suptitle("Provincial GDP level and growth by scenario — British Columbia highlighted",
             fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "pit_gdp_by_scenario.png", dpi=120)
plt.show()

## II.8 — The upstream sample's own outputs, reproduced

The strongest evidence that the addon does not disturb the pipeline is that the pipeline's
**standard artifacts still come out**. This section reproduces
`Sample_macroabm-Canada_provincial_run.ipynb`: the same HDF5 results file, read back with the
same utilities, and the same plots — but from a run with progressive PIT **on**.

Nothing below is tax-specific. If it looks like the sample notebook, that is the point.

> **Mind the size.** `sim.save()` writes every province's household-level series plus
> rest-of-world — roughly **6.7 GB** for a 16-step run at `SCALE = 1000` (ROW alone is 2.4 GB;
> the provincial `households` groups another ~5 GB). This notebook writes the **full** file by
> default, because the model is not a BC-only model and a truncated artifact would discard the
> nine provinces scenario 03 exists to demonstrate.
>
> If disk or transfer size is a constraint, set `SAVE_FULL_H5 = False` in the scenario cell.
> That writes BC in full plus an aggregated `CAN` group instead — about 20× smaller. The `CAN`
> group sums **extensive** quantities only (output, spending, revenue, production); intensive
> ones (CPI, unemployment, price indices) are deliberately excluded, because summing an index
> across provinces is meaningless and such a column is exactly what gets mistaken for a
> national figure.
>
> The plots below read whatever provinces the file happens to contain, so they work either
> way. The **danger zone at the end of the notebook deletes this file.**

In [ ]:
# --- the sample notebook's results utilities -------------------------------------
from typing import Any, Dict, List, Optional

import h5py
import pandas as pd

H5_FILE = OUTPUT_DIRECTORY / H5_FILENAME
INDUSTRY_NAMES_CSV = INPUT_PATH / "can_industries_wnames.csv"


def load_industry_mapping():
    if not INDUSTRY_NAMES_CSV.exists():
        return {}
    df = pd.read_csv(INDUSTRY_NAMES_CSV)
    return dict(zip(df["Firm_ID"], df["Industry_Name"]))


def _build_tree(group):
    tree = {}
    for key in group.keys():
        obj = group[key]
        if isinstance(obj, h5py.Dataset):
            tree[key] = {"type": "dataset"}
        else:
            tree[key] = {"type": "group", "children": _build_tree(obj)}
    return tree


def get_tree(file_path):
    with h5py.File(file_path, "r") as f:
        return _build_tree(f)


def list_dataset_paths(tree, prefix=""):
    paths = []
    for key in sorted(tree):
        entry = tree[key]
        current = f"{prefix}/{key}" if prefix else key
        if entry["type"] == "dataset":
            paths.append(current)
        else:
            paths.extend(list_dataset_paths(entry["children"], current))
    return paths


def load_dataset(file_path, dataset_path):
    with h5py.File(file_path, "r") as f:
        return f[dataset_path][()]


def compute_real_gdp(file_path, province):
    gdp_path, cpi_path = f"{province}/economy/gdp_output", f"{province}/economy/cpi"
    with h5py.File(file_path, "r") as f:
        if gdp_path not in f or cpi_path not in f:
            return None
        gdp = f[gdp_path][()].astype(float)
        cpi = f[cpi_path][()].astype(float)
    return np.divide(gdp, cpi, out=np.full_like(gdp, np.nan), where=cpi != 0)


def get_column_labels(file_path, dataset_path, industry_mapping=None):
    columns_path = f"{dataset_path}_columns"
    with h5py.File(file_path, "r") as f:
        if columns_path not in f:
            return None
        cols = np.asarray(f[columns_path][()])
    if cols.ndim == 1:
        if industry_mapping:
            return [industry_mapping.get(int(i), str(int(i))) for i in cols]
        return [str(int(i)) for i in cols]
    return [str(v) for v in cols.flatten()]


def sum_regions(file_path, province_keys, tail_path):
    total = None
    with h5py.File(file_path, "r") as f:
        for province in province_keys:
            full = f"{province}/{tail_path}"
            if full not in f:
                continue
            data = f[full][()].astype(float)
            total = data.copy() if total is None else total + data
    return total


industry_mapping = load_industry_mapping()
tree = get_tree(H5_FILE)
regions = sorted(k for k, v in tree.items()
                 if v["type"] == "group" and k.startswith("CAN_"))
all_paths = list_dataset_paths(tree)

print(f"HDF5 results file : {H5_FILE.name}  ({H5_FILE.stat().st_size / 1e6:,.0f} MB)")
print(f"industry mapping  : {len(industry_mapping)} industries")
print(f"top-level groups  : {sorted(k for k, v in tree.items() if v['type'] == 'group')}")
print(f"provinces in file : {regions}")
print(f"datasets          : {len(all_paths):,}")
print()
print("Same artifact, same utilities, same structure as the sample notebook.")

In [ ]:
# --- Sample plot 1: nominal GDP by province (level + growth), from the HDF5 ------
fig, axes = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
with h5py.File(H5_FILE, "r") as f:
    for region in regions:
        path = f"{region}/economy/gdp_output"
        if path in f:
            series = f[path][()].astype(float).ravel()
            is_bc = region == "CAN_BC"
            style = dict(linewidth=2.4 if is_bc else 1.0,
                         color=BC_COLOUR if is_bc else None,
                         alpha=1.0 if is_bc else 0.75, zorder=3 if is_bc else 2)
            axes[0].plot(series, label=PROV_LABELS.get(region, region), **style)
            axes[1].plot(range(1, len(series)), qoq_growth(series), **style)
axes[1].axhline(0.0, color="black", linewidth=0.7, linestyle="--")
axes[0].set_title("Nominal GDP output by province  (scenario 02: BC progressive)")
axes[0].set_ylabel("Nominal GDP output (model units)")
axes[1].set_ylabel("Growth, quarter-on-quarter (%)")
axes[1].set_xlabel("Timestep (3 months)")
axes[0].legend(fontsize=7, ncol=2)
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "sample_gdp_by_province.png", dpi=120)
plt.show()

In [ ]:
# --- Sample plot 2: real GDP by province (GDP / CPI), level + growth -------------
# compute_real_gdp is the sample notebook's own helper, unmodified.
fig, axes = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
for region in regions:
    real_gdp = compute_real_gdp(H5_FILE, region)
    if real_gdp is not None:
        series = real_gdp.ravel()
        is_bc = region == "CAN_BC"
        style = dict(linewidth=2.4 if is_bc else 1.0,
                     color=BC_COLOUR if is_bc else None,
                     alpha=1.0 if is_bc else 0.75, zorder=3 if is_bc else 2)
        axes[0].plot(series, label=PROV_LABELS.get(region, region), **style)
        axes[1].plot(range(1, len(series)), qoq_growth(series), **style)
axes[1].axhline(0.0, color="black", linewidth=0.7, linestyle="--")
axes[0].set_title("Real GDP by province (GDP / CPI)  (scenario 02: BC progressive)")
axes[0].set_ylabel("Real GDP (model units)")
axes[1].set_ylabel("Growth, quarter-on-quarter (%)")
axes[1].set_xlabel("Timestep (3 months)")
axes[0].legend(fontsize=7, ncol=2)
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "sample_real_gdp_by_province.png", dpi=120)
plt.show()

### Firm prices across scenarios *(mirrors the scenario-comparison sample)*

`Sample_macroabm-CANADA_run_time_iteration.ipynb` compares firm prices across its scenarios
and plots the difference. The same panels, for ours, using BC's good-price index.

> **The difference panel is noise-dominated — read it as machinery, not as a result.** Per
> II.6, cross-scenario differences in this model contain shared-RNG wander as large as any tax
> effect. The panel shows the scenario-comparison flow works with the addon in place; it does
> **not** establish that progressive PIT moved prices.

In [ ]:
# --- Sample-B style: price levels per scenario, and the difference ---------------
ind_labels = [industry_mapping.get(i, str(i)) for i in range(n_industries)]
price_diff = res_02["ts_price"] - res_01["ts_price"]
top_idx = np.argsort(np.abs(price_diff).mean(axis=0))[::-1][:12]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, res in zip(axes[:2], [res_01, res_02]):
    for i in top_idx:
        ax.plot(x, res["ts_price"][:, i], label=ind_labels[i], linewidth=1.2)
    ax.set_title(f"{res['label']}\nBC firm prices (top 12 by |diff|)", fontsize=9)
    ax.set_xlabel("Timestep (3 months)")
    ax.set_ylabel("Price index (2014 = calibrated)")
    ax.legend(fontsize=6, ncol=2)
    ax.grid(True, alpha=0.3)

ax = axes[2]
for i in top_idx:
    ax.plot(x, price_diff[:, i], label=ind_labels[i], linewidth=1.2)
ax.axhline(0.0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Difference (02 - 01)\nNOISE-DOMINATED - see II.6, not a tax result", fontsize=9)
ax.set_xlabel("Timestep (3 months)")
ax.set_ylabel("Price difference")
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

plt.suptitle("BC firm prices by scenario", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIRECTORY / "sample_firm_prices.png", dpi=120)
plt.show()

## II.9 — Known limitations

The addon is calibrated for **British Columbia**. Where other provinces appear (scenario 03)
they demonstrate that the *mechanism* routes each jurisdiction to its own schedule — they are
not calibrated tax levels and should not be read as such.

- **BC's modelled effective rate is comparable to BC's actual**, sitting modestly below it.
  Two deliberate omissions offset one another: not all income streams are in the taxable pool
  (which suppresses the rate), and only 4 of BC's 14 non-refundable credit kinds are implemented
  (which inflates it). Completing one side alone would make the fit *worse*, so the level
  should be re-anchored whenever either is extended.
- **Ontario and PEI are understated.** Both levy a **surtax on the tax amount** rather than on
  income (Ontario: 20% of basic tax above one threshold plus a further 36% above a second),
  and Ontario also has an income-tested Health Premium. Neither is expressible as a marginal
  income bracket, so neither is in the schedule.
- **Credits are valued at the bottom bracket rate, not the schedule's `rate` column.**
  The runtime multiplies each credit base by the jurisdiction's lowest marginal rate. For nine
  of the eleven jurisdictions those coincide, so nothing differs; **Quebec 2014-2016 is the one
  exception** (a 20% statutory conversion rate against a 16% bottom bracket), which makes
  Quebec's credits ~20% too small in those years — they coincide from 2017 on. (Income-testing
  itself *is* applied: the Age Amount phases out between `clawback` and `top`, and the Spousal
  and eligible-dependant amounts are reduced by the spouse's or dependant's income
  above the published exemption.)
- **Build-to-build reproducibility is under investigation in the base model.** Repeated
  builds at a fixed configuration do not always yield an identical synthetic population, and
  BC's effective rate moves accordingly — by more than the seed-to-seed noise within a single
  build. This is a base-model matter, not something the tax addon introduces, and it is being
  worked on. The practical consequence for now: **treat any single effective-rate measurement
  as indicative, and calibrate against several builds rather than one.**
- **Macro aggregates cannot resolve a tax effect at a single seed.** The end-of-run table in
  II.7 prints each series' tax effect beside a noise floor measured on this same run, and every
  effect sits inside its floor. The figures move with the seed and the build, so they are
  printed from the run in front of you rather than quoted here — a number frozen into this
  text goes stale the next time anything changes. The macro panels in II.7 are therefore
  **context, not evidence**; resolving
  the effect would need trial averaging, as the upstream scenario notebook does with
  `n_trials`. This is a property of the base model (BC carries ~1,900 synthetic households at
  `scale = 1000`), not of the addon. The **effective rate and PIT revenue** are unaffected by
  this — they move deterministically and by a wide margin, and are what the notebook's claims
  rest on.
- **Cross-scenario GDP paths are not isolated** — see II.6. The shared global RNG means
  provinces move between scenarios for non-tax reasons. Judge tax effects by the effective
  rate and PIT revenue, not by GDP differences.
- **The results file is large and contains NaN — both upstream, neither from the addon.**
  `sim.save()` writes ~**6.7 GB** at `SCALE = 1000` (rest-of-world alone is 2.4 GB; the
  provincial `households` groups another ~5 GB), and it grows with the agent count. It also
  carries **NaN in roughly 291 datasets per province** — across households, firms, banks,
  government entities and parts of `economy` — about 40 million non-finite values in total,
  mostly in per-agent arrays such as `households/nominal_amount_spent_in_lcu*` (~6% NaN).
  The sample notebook's own results file has the same. **The series this notebook plots are
  clean**: `gdp_output`, `cpi`, `firms/production` and `central_government/taxes_income` are
  finite for every province. Anything else read from the file should be checked before use.
- **A pickle built by older model code will fail, not run wrong.** A fix that changes the data
  layer — the social-housing rent exclusion, say — invalidates existing pickles, and the
  restored exact GDP identity check rejects them *inside* `Simulation.from_datawrapper` with a
  message that reads like a model bug. The guard in II.2 catches this by constructing a
  throwaway `Simulation` and rebuilding, rather than trusting that the file merely exists.
  If a pickle is ever reused across a model change without that guard, expect a confusing
  `AssertionError: mismatch, output/income GDP`.
- **Upstream warnings are suppressed in I.1** — chiefly non-finite-value `RuntimeWarning`s
  (`invalid value encountered`, from the technical-coefficient growth path and from IO/SEA
  calibration of sparse provincial sectors), plus benign pandas dtype/fragmentation notices.
  **Not** from the tax addon: they fire with the taxation data entirely absent, and they do
  not reach the reported results — GDP, CPI and production are finite for every province. The
  filters are there for readability; delete them to see the warnings.

None of these touch the parity claim in II.5, which is structural and exact.

## II.10 — Summary

- **The upstream flow is unchanged.** Part I is the sample provincial notebook. The data
  configuration says nothing about taxation, and a missing taxation tree degrades cleanly to
  the flat model.
- **Parity is exact.** With every government opted out, the model is bit-for-bit identical to
  one with no taxation data at all — `0.000000e+00` across every province — even though the
  full schedules are attached. `activate_taxation` returns the configuration untouched.
- **Opt-in is per government, and coverage follows the data.** One flag per government
  decides who is taxed progressively; the schedule files decide who *can* be. No province is
  ever taxed on another's schedule.

The addon is a layer on top of the model rather than a change to it — which is what makes it
mergeable upstream without disturbing existing behaviour.

---
## ⚠️ Danger zone — delete this notebook's outputs

Removes the artifacts **this notebook** produced. It will **not** touch anything else in
`output/` — the sample notebooks write their own pickle and HDF5 results there, and those are
left alone.

Two separate switches, because they cost very different amounts to undo:

| | Cost to regenerate |
|---|---|
| **Figures** (`pit_*.png`, `sample_*.png`) | seconds — re-run the plot cells (II.7–II.8) |
| **The data pickles** | **minutes each** — a full rebuild from raw data |

`DELETE_PICKLE` removes **every pickle this notebook has built, at every `SCALE`** — not only
the one for the current setting. If you have experimented with `SCALE`, that is several files
and a substantial rebuild. It is **off by default**, and the next cell lists exactly what
exists — and exactly what will be left alone — before anything is touched.

Nothing is removed until you set `CONFIRM = True` in that cell, re-run it, and then run the
cell below it.

In [ ]:
# --- what this notebook produced; nothing else in output/ is considered -----------
FIGURES = [
    OUTPUT_DIRECTORY / "pit_bc_headline.png",
    OUTPUT_DIRECTORY / "pit_bc_macro.png",
    OUTPUT_DIRECTORY / "pit_effective_rate.png",
    OUTPUT_DIRECTORY / "pit_gdp_by_scenario.png",
    # written by earlier versions of this notebook; safe to clear
    OUTPUT_DIRECTORY / "pit_gdp_difference.png",
    OUTPUT_DIRECTORY / "pit_revenue.png",
    OUTPUT_DIRECTORY / "sample_gdp_by_province.png",
    OUTPUT_DIRECTORY / "sample_real_gdp_by_province.png",
    OUTPUT_DIRECTORY / "sample_firm_prices.png",
    OUTPUT_DIRECTORY / H5_FILENAME,
]

DELETE_PICKLE = False        # <- set True to delete EVERY pickle, at EVERY scale
CONFIRM       = False        # <- set True to arm the deletion, then run the cell below


# Every pickle this notebook has ever built, at ANY scale -- not merely the one for the
# current SCALE. Experimenting with SCALE leaves several behind, and deleting only the
# current PKL_PATH would silently strand the rest. The glob is deliberately narrow: it
# matches this notebook's own naming and cannot touch the sample notebooks' pickles.
def _pickles():
    return sorted(OUTPUT_DIRECTORY.glob("data_provincial_pit_all_scale*.pkl"))


def _targets():
    targets = [p for p in FIGURES if p.exists()]
    if DELETE_PICKLE:
        targets.extend(_pickles())
    return targets


def _delete():
    targets = _targets()
    if not targets:
        print("Nothing to delete.")
        return
    for p in targets:
        p.unlink()
        print(f"  deleted  {p.name}")
    print(f"\n{len(targets)} file(s) removed. Everything else in output/ is untouched.")


targets = _targets()
print("These files WOULD be deleted:")
for p in targets:
    mb = p.stat().st_size / 1e6
    if p.suffix == ".pkl":
        # SCALE=1000 rebuilds in ~2 min on a quiet machine; build time is CPU-bound
        # and machine-dependent (a busy machine takes longer). A smaller SCALE has more
        # agents to synthesise, so it takes longer still.
        sc = p.stem.split("scale")[-1]
        tag = ("   <-- ~2 min to rebuild" if sc == "1000"
               else "   <-- longer to rebuild (smaller SCALE = more agents)")
    else:
        tag = ""
    print(f"   {p.name:<42} {mb:>7.2f} MB{tag}")
if not targets:
    print("   (none)")

kept = sorted(p.name for p in OUTPUT_DIRECTORY.glob("*")
              if p.is_file() and p not in targets)
print(f"\nThese files in output/ will be LEFT ALONE ({len(kept)}):")
for name in kept:
    print(f"   {name}")

if not DELETE_PICKLE:
    existing = _pickles()
    print(f"\nPROTECTED - DELETE_PICKLE is False. {len(existing)} pickle(s) kept:")
    for p in existing:
        print(f"   {p.name:<42} {p.stat().st_size / 1e6:>7.2f} MB")
    if not existing:
        print("   (none built yet)")

In [ ]:
# --- delete ----------------------------------------------------------------------
# Nothing is removed unless CONFIRM was set to True in the cell above.
if CONFIRM:
    _delete()
else:
    print("CONFIRM is False - nothing deleted.")
    print("To remove the files listed above: set CONFIRM = True in the cell above, "
          "re-run it, then re-run this cell.")